In [2]:
import os
import json
import pandas as pd
import re
import numpy as np


# Connect to Google Drive
import gspread
import gspread_dataframe
from google.oauth2.service_account import Credentials
from google.oauth2 import service_account
from googleapiclient.discovery import build
from gspread_dataframe import set_with_dataframe
from gspread_dataframe import get_as_dataframe

In [ ]:
# 1. Fetch credentials from environment variable
creds_env = os.environ.get("GDRIVE_CREDENTIALS_KC")

if not creds_env:
    raise ValueError("Environment variable 'GDRIVE_CREDENTIALS' was not found.")

creds_json = json.loads(creds_env)

# 2. Define required scopes
scopes = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive",
]

# 3. Authenticate service account
creds = service_account.Credentials.from_service_account_info(
    creds_json, scopes=scopes
)

# 4. Initialize Google API clients
drive_service = build("drive", "v3", credentials=creds)
sheets_service = build("sheets", "v4", credentials=creds)

gc = gspread.authorize(creds)

print("Google Drive and Sheets services successfully initialized.")

Mounted at /content/drive


In [17]:
# Open files

# Open organic posts file
forms_data = gc.open_by_key('1PGajyPdI45WPENpWdRK3eYFFbaHIPiKvzZ3uVtErcTg')
forms_data = forms_data.get_worksheet(0)
forms_data = get_as_dataframe(forms_data)


posts_data = gc.open_by_key('1R6b2vfc_UyFmsOuiZBm6Y5b824MQRnH4nh2eM__NgNo')
posts_data = posts_data.worksheet('data_profile_post_max')
posts_data = get_as_dataframe(posts_data)

posts_comments = gc.open_by_key('1MpdbGBD2YS2-J2tOTFqPW2tDwUzW-1H6QNblyxuJTgw')
posts_comments = posts_comments.get_worksheet(0)
posts_comments = get_as_dataframe(posts_comments)




In [18]:
# Clean databases
forms_data = forms_data.fillna(0)
posts_data = posts_data.fillna(0)
posts_comments = posts_comments.fillna(0)

/tmp/ipykernel_2294/3519262252.py:4: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  posts_comments = posts_comments.fillna(0)


In [19]:
# Estimate sentimment for each published post
sentiment_posts = posts_comments.groupby(['post_url', 'perfil']).agg(
    positive_count=('sentimento_nps', lambda x: (x == 'promotor').sum()),
    negative_count=('sentimento_nps', lambda x: (x == 'detrator').sum()),
    neutral_count=('sentimento_nps', lambda x: (x == 'neutro').sum())
).reset_index()

# 2. Sum the three sentiment columns to get the total
sentiment_posts['total_sentiments'] = (
    sentiment_posts['positive_count'] +
    sentiment_posts['negative_count'] +
    sentiment_posts['neutral_count']
)

# 3. Calculate the percentage of each sentiment over the total
sentiment_posts['positive_percentage'] = sentiment_posts['positive_count'] / sentiment_posts['total_sentiments']
sentiment_posts['negative_percentage'] = sentiment_posts['negative_count'] / sentiment_posts['total_sentiments']
sentiment_posts['neutral_percentage'] = sentiment_posts['neutral_count'] / sentiment_posts['total_sentiments']

In [20]:
# Prepare datasets for join

posts_data = posts_data.rename(columns={'username_shared': 'influencer'})
forms_data = forms_data.rename(columns={'Link of Post': 'url'})
sentiment_posts = sentiment_posts.rename(columns={'post_url': 'url'})

In [21]:
# Join datasets
instagram_influencers_final = posts_data
instagram_influencers_final = instagram_influencers_final.merge(forms_data, on='url', how='left')
instagram_influencers_final = instagram_influencers_final.merge(sentiment_posts, on='url', how='left')
instagram_influencers_final

,run_datetime,Plataform_x,username,influencer,followers_count,following_count,total_posts_count,code,taken_at,url,...,"Objetivo: alcance, interacción, etc",Budget: AON / campaña,perfil,positive_count,negative_count,neutral_count,total_sentiments,positive_percentage,negative_percentage,neutral_percentage
0,2026-08-26 8:59:14,Instagram,crispierri,crispierri,958367.0,878.0,560.0,DU6r1KWia0X,2026-02-18 20:01:53,https://www.instagram.com/reel/DU6r1KWia0X/,...,0.0,0.0,crispierri,9,10,43,62,0.145161,0.161290,0.693548
1,2026-08-26 9:00:09,Instagram,pablobruschi,pablobruschi,1720311.0,889.0,2293.0,DU6k9NlEUXf,2026-02-18 19:02:46,https://www.instagram.com/reel/DU6k9NlEUXf/,...,0.0,0.0,pablobruschi,16,23,87,126,0.126984,0.182540,0.690476
2,2026-08-26 9:02:02,Instagram,benjacalero01,benjacalero01,1178952.0,817.0,1023.0,DU6hr61AZQz,2026-02-18 18:33:09,https://www.instagram.com/reel/DU6hr61AZQz/,...,0.0,0.0,benjacalero01,9,7,36,52,0.173077,0.134615,0.692308
3,2026-08-26 9:03:08,Instagram,losariasbrothers,losariasbrothers,1331445.0,423.0,529.0,DU6hf00jCLX,2026-02-18 18:31:31,https://www.instagram.com/reel/DU6hf00jCLX/,...,0.0,0.0,losariasbrothers,7,2,18,27,0.259259,0.074074,0.666667
4,2026-08-26 9:03:49,Instagram,lucaslezin,lucaslezin,1021975.0,969.0,1401.0,DU6pEotkdMv,2026-02-18 19:39:05,https://www.instagram.com/reel/DU6pEotkdMv/,...,0.0,0.0,lucaslezin,18,16,78,112,0.160714,0.142857,0.696429
5,2026-08-26 9:05:04,Instagram,kuki_youtube,kuki_youtube,869777.0,459.0,66.0,DU6eQlwkntp,2026-02-18 18:03:39,https://www.instagram.com/reel/DU6eQlwkntp/,...,0.0,0.0,kuki_youtube,52,16,86,154,0.337662,0.103896,0.558442
6,2026-08-26 8:58:38,Instagram,_dulcepink_,_dulcepink_,1070279.0,1219.0,327.0,DU6UHhykfyC,2026-02-18 16:36:10,https://www.instagram.com/reel/DU6UHhykfyC/,...,0.0,0.0,_dulcepink_,10,25,37,72,0.138889,0.347222,0.513889
7,2026-08-26 9:07:24,Instagram,caam.prz,caam.prz,131739.0,1164.0,258.0,DU6DH6FgNSn,2026-02-18 14:06:26,https://www.instagram.com/reel/DU6DH6FgNSn/,...,0.0,0.0,caam.prz,3,8,31,42,0.071429,0.190476,0.738095
8,2026-08-26 9:08:05,Instagram,gastiobeide,gastiobeide,507729.0,1141.0,960.0,DU6XHeXktGp,2026-02-18 17:02:30,https://www.instagram.com/reel/DU6XHeXktGp/,...,0.0,0.0,gastiobeide,10,22,63,95,0.105263,0.231579,0.663158
9,2026-08-26 9:09:25,Instagram,_dulcepink_,_dulcepink_,1070276.0,1219.0,327.0,DVWI6gUDUkk,2026-03-01 11:55:00,https://www.instagram.com/reel/DVWI6gUDUkk/,...,0.0,0.0,_dulcepink_,27,80,39,146,0.184932,0.547945,0.267123


In [22]:
# Agregar código orgánico
instagram_influencers_final["Organic_ID"] = instagram_influencers_final["code"]


In [26]:
# Rename columns
instagram_influencers_final = instagram_influencers_final.rename(columns={'post_caption': 'copy'})
instagram_influencers_final = instagram_influencers_final.rename(columns={'Plataform_x': 'platform'})
instagram_influencers_final = instagram_influencers_final.rename(columns={'taken_at': 'date_published'})

instagram_influencers_final = instagram_influencers_final.rename(columns={'media_type': 'format'})
instagram_influencers_final = instagram_influencers_final.rename(columns={'play_count': 'views'})
instagram_influencers_final = instagram_influencers_final.rename(columns={'comment_count': 'comments'})
instagram_influencers_final = instagram_influencers_final.rename(columns={'like_count': 'likes'})
instagram_influencers_final = instagram_influencers_final.rename(columns={'Marca': 'brand'})
instagram_influencers_final = instagram_influencers_final.rename(columns={'followers_count': 'followers'})


In [28]:
# Crear columnas
instagram_influencers_final["shares"] = 0
instagram_influencers_final["saves"] = 0
instagram_influencers_final["content_type"] = "Influencers"
instagram_influencers_final["total_interactions"] = instagram_influencers_final["likes"] + instagram_influencers_final["comments"] + instagram_influencers_final["shares"] + instagram_influencers_final["saves"]
instagram_influencers_final["engagement_rate"] = (
    instagram_influencers_final["total_interactions"]
    .div(instagram_influencers_final["views"])
    .replace([np.inf, -np.inf], 0)
    .fillna(0)
)
instagram_influencers_final["positive_comments"] = instagram_influencers_final["positive_percentage"] * instagram_influencers_final["comments"]
instagram_influencers_final["negative_comments"] = instagram_influencers_final["negative_percentage"] * instagram_influencers_final["comments"]
instagram_influencers_final["neutral_comments"] = instagram_influencers_final["neutral_percentage"] * instagram_influencers_final["comments"]



In [30]:
# Seleccionar y ordenar columnas

cols = [
    "url",
    "copy",
    "date_published",
    "platform",
    "format",
    "influencer",
    "Country",
    "Organic_ID",
    "brand",
    "content_type",
    "views",
    "likes",
    "comments",
    "shares",
    "saves",
    "total_interactions",
    "engagement_rate",
    "positive_percentage",
    "negative_percentage",
    "neutral_percentage",
    "positive_comments",
    "negative_comments",
    "neutral_comments",
    "run_datetime",
    "followers"
]

instagram_influencers_final = instagram_influencers_final[cols]
instagram_influencers_final = instagram_influencers_final.reindex(columns=cols)

In [31]:
# Adjust date
instagram_influencers_final["date_published"] = pd.to_datetime(
    instagram_influencers_final["date_published"],
    format="mixed",
    errors="coerce",
).dt.date

instagram_influencers_final["run_datetime"] = pd.to_datetime(
    instagram_influencers_final["run_datetime"],
    format="mixed",
    errors="coerce",
).dt.date

In [32]:
# Force missing likes (-1) to 0
instagram_influencers_final["likes"] = instagram_influencers_final["likes"].clip(lower=0)

In [33]:
# Save final table
# Open the destination sheets file
sh = gc.open_by_key('1KtpTF1FTzh-hsb1V0duckm-pmQ9yjvFC-KMivyqXGCc')
worksheet = sh.get_worksheet(0)

# Replace old data with new data
set_with_dataframe(worksheet, instagram_influencers_final)
print("DataFrame saved successfully!")

DataFrame saved successfully!
